In [2]:
import duckdb

In [3]:
duckdb.read_csv("../raw_data/google_ads/*.csv")

┌─────────────┬─────────────┬──────────────────────┬─────────────┬───────────┬─────────────┬────────┬──────────────┐
│    date     │ campaign_id │    campaign_name     │   channel   │ spend_usd │ impressions │ clicks │ soft_deleted │
│   varchar   │    int64    │       varchar        │   varchar   │  double   │    int64    │ int64  │   boolean    │
├─────────────┼─────────────┼──────────────────────┼─────────────┼───────────┼─────────────┼────────┼──────────────┤
│ 30-Mar-2025 │           1 │ Brand Awareness Q1   │ Search      │    281.27 │       31784 │    814 │ false        │
│ 2025-03-30  │           5 │ New User Acquisition │ Search      │     299.1 │       31775 │   1441 │ false        │
│ 2025-03-30  │          21 │ Dynamic Search Ads   │ Paid Search │    238.37 │       43935 │   2773 │ false        │
│ 03/31/2025  │           1 │ Brand Awareness Q1   │ SEARCH      │    317.87 │       39921 │   1402 │ false        │
│ 31-Mar-2025 │           5 │ New User Acquisition │ Paid Search

In [4]:
conn = duckdb.connect("marketing_data.duckdb")

conn.execute(""" CREATE SCHEMA IF NOT EXISTS raw""")

conn.execute("""
    CREATE TABLE IF NOT EXISTS raw.google_ads AS
    SELECT * FROM read_csv('../raw_data/google_ads/*.csv')
""")

In [5]:
conn.execute("show all tables").fetchdf()

,database,schema,name,column_names,column_types,temporary
0,marketing_data,raw,google_ads,"[date, campaign_id, campaign_name, channel, sp...","[VARCHAR, BIGINT, VARCHAR, VARCHAR, DOUBLE, BI...",False


In [6]:
conn.execute("select * from raw.google_ads limit 10").fetchdf()

,date,campaign_id,campaign_name,channel,spend_usd,impressions,clicks,soft_deleted
0,30-Mar-2025,1,Brand Awareness Q1,Search,281.27,31784,814,False
1,2025-03-30,5,New User Acquisition,Search,299.10,31775,1441,False
2,2025-03-30,21,Dynamic Search Ads,Paid Search,238.37,43935,2773,False
3,03/31/2025,1,Brand Awareness Q1,SEARCH,317.87,39921,1402,False
4,31-Mar-2025,5,New User Acquisition,Paid Search,509.65,74533,5072,False
5,2025-03-31,21,Dynamic Search Ads,SEARCH,167.15,19176,298,False
6,01-Apr-2025,1,Brand Awareness Q1,Paid Search,287.15,30191,866,False
7,01-Apr-2025,5,New User Acquisition,search,335.86,37748,2822,False
8,01-Apr-2025,21,Dynamic Search Ads,SEARCH,173.32,19434,481,False
9,02-Apr-2025,1,Brand Awareness Q1,SEARCH,421.91,47163,1285,False


In [7]:
conn.execute("select count(*) from raw.google_ads").fetchdf()

,count_star()
0,2092


In [8]:
conn.execute("select campaign_name, count(*) from raw.google_ads group by campaign_name").fetchdf()

,campaign_name,count_star()
0,Brand Awareness Q1,18
1,Summer Sale - Retargeting,125
2,Summer Sale - Retargeting,8
3,Summer Sale - Retargeting,1
4,Dynamic Search Ads,2
5,Brand Awareness Q1,1
6,New User Acquisition,19
7,YouTube Bumpers,190
8,App Install Campaign,1
9,Dynamic Search Ads,315


In [9]:
conn.execute("select distinct channel from raw.google_ads").fetchdf()

,channel
0,SHOPPING
1,Display
2,video
3,Paid Search
4,product_shopping
5,search
6,SEARCH
7,Video
8,display
9,DISPLAY


In [14]:
duckdb.read_json("../raw_data/segment/segment_tracks.jsonl")

┌──────────────────────────────────┬─────────┬────────────────┬─────────────────────────┬─────────┬──────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬─────────────────────────────────────────────────────────────────────────────────────┐
│            message_id            │  type   │     event      │        timestamp        │ user_id │ anonymous_id │                                                                                                                                   properties                                                                                                                                   │                                       context                                       │
│             varchar              │ varchar │    va

In [30]:
conn.execute("""
    CREATE TABLE IF NOT EXISTS raw.segment_tracks AS
    SELECT 
        message_id,
        type,
        event,
        timestamp,
        user_id,
        anonymous_id,
        json(properties) as properties,
        json(context) as context,
        session_id
             
    FROM read_json('../raw_data/segment/segment_tracks.jsonl', sample_size=-1)

""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

**sample_size=-1 to force DuckDB to scan the entire file before inferring the schema**



[DuckDB Loading JSON Documentation](https://duckdb.org/docs/current/data/json/loading_json)

In [31]:
conn.execute("describe raw.segment_tracks").fetchdf()

,column_name,column_type,null,key,default,extra
0,message_id,VARCHAR,YES,None,None,None
1,type,VARCHAR,YES,None,None,None
2,event,VARCHAR,YES,None,None,None
3,timestamp,VARCHAR,YES,None,None,None
4,user_id,VARCHAR,YES,None,None,None
5,anonymous_id,VARCHAR,YES,None,None,None
6,properties,JSON,YES,None,None,None
7,context,JSON,YES,None,None,None
8,session_id,VARCHAR,YES,None,None,None


In [32]:
conn.execute("select * from raw.segment_tracks where message_id = 'a55f626bb248f948693a07b3c16a4403' ").fetch_df()

,message_id,type,event,timestamp,user_id,anonymous_id,properties,context,session_id
0,a55f626bb248f948693a07b3c16a4403,track,product_view,2025-03-30 11:03:56 UTC,u_04441,anon_292484,"{""campaign_id"":""8"",""channel"":""display"",""page_u...","{""library_name"":""analytics.js"",""library_versio...",None


In [33]:

conn.execute("select * from raw.segment_tracks where message_id = '56e5616e27be631a9130ae514fe7b818' ").fetch_df()

,message_id,type,event,timestamp,user_id,anonymous_id,properties,context,session_id
0,56e5616e27be631a9130ae514fe7b818,track,page_view,2025-06-28 02:52:16 UTC,u_00612,anon_836518,"{""campaign_id"":""16"",""channel"":""search"",""page_u...","{""library_name"":""analytics.js"",""library_versio...",sess_32372495
